Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise: Getting Comfortable with Our Open Lakehouse

Verify your container works and that Iceberg tables are reachable from multiple engines.

**The stack:**
- **Apache Iceberg** — open table format
- **MinIO** — S3-compatible storage
- **Apache Polaris** — Iceberg REST catalog
- **Spark / Trino** — two query engines sharing the same tables

⚠️ This environment uses **Spark Connect**: one Spark server runs in the background, notebooks connect as thin clients. If you see `ConnectionRefusedError` when starting the session, the server isn't up — check `docker logs jupyter-spark`, or restart with `docker compose restart jupyter`.

## Explore the Environment


### Jupyter Notebooks

You're here. Lesson notebooks live in `/notebooks`; the file browser on the left shows them. Use the integrated terminal if you need shell access.

### Polaris Catalog

**URL:** UI not yet released.

Polaris is an Iceberg REST catalog. Engines (Spark, Trino, Flink) call it over HTTP to resolve where a table's metadata lives. Separating the catalog from the engines is what lets multiple engines share the same tables.

### Spark Configuration

**File:** `/home/jovyan/.sparkconf/spark-defaults.conf`

Generated by Docker Compose on startup. It wires the catalog (Polaris), storage (MinIO S3), and required packages (Iceberg runtime + AWS bundle). The cell below prints it.

In [1]:
with open('/home/jovyan/.sparkconf/spark-defaults.conf', 'r') as f:
    config = f.read()
    print(config)

# Copyright 2026 Snowflake Inc.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Iceberg Spark Dependencies
spark.jars.packages=org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0,software.amazon.awssdk:bundle:2.20.18,software.amazon.awssdk:url-connection-client:2.20.18,org.apache.hadoop:hadoop-aws:3.4.1

# Iceberg Spark SQL/Catalyst Extensions
spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSpark

## Initialize Spark Session

Create a session that connects to the pre-configured environment. The Polaris catalog is already wired in.

In [2]:
from pyspark.sql import SparkSession

# All configuration is loaded from spark-defaults.conf
spark = SparkSession.builder \
    .appName("OpenLakehouse") \
    .getOrCreate()

print(f" Spark {spark.version} initialized!")
print(f" Default Catalog: {spark.conf.get('spark.sql.defaultCatalog', 'spark_catalog')}")

/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

 Spark 4.0.1 initialized!
 Default Catalog: polaris


## Create Your First Iceberg Table

Create a namespace (like a database schema) and a table inside it.

In [3]:


# Create namespace (like a schema in traditional databases)
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.demo")
print(" Namespace 'demo' created!")

# Drop and recreate so re-running the notebook starts fresh
spark.sql("DROP TABLE IF EXISTS polaris.demo.employees")

# Create an Iceberg table
# USING iceberg tells Spark to create an Iceberg table rather than a plain
# Spark-managed table. Without it, Spark creates a non-Iceberg table that
# doesn't support snapshots, time travel, or schema evolution.
#
# Format versions: V1 = original spec, V2 = row-level deletes,
# V3 = delete vectors and new types like Variant and Geo.
# V1/V2 are still in wide use. V3 is recommended for new tables.
# Upgrading is a one-way metadata change with no data rewrite.
spark.sql("""
    CREATE TABLE polaris.demo.employees (
        id INT,
        name STRING,
        department STRING
    ) USING iceberg
    TBLPROPERTIES (
        'format-version' = '3'
    )
""")
print(" Table 'employees' created!")

 Namespace 'demo' created!
 Table 'employees' created!


## Insert Sample Data

In [4]:
spark.sql("""
    INSERT INTO polaris.demo.employees VALUES
    (1, 'Alice', 'Engineering'),
    (2, 'Bob', 'Sales'),
    (3, 'Charlie', 'Marketing'),
    (4, 'Diana', 'Engineering')
""")
print(" Sample data inserted!")

 Sample data inserted!


## Query with Spark

Read the data back.

In [5]:
spark.sql("""
    SELECT * FROM polaris.demo.employees
""").show()

+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  1|  Alice|Engineering|
|  2|    Bob|      Sales|
|  3|Charlie|  Marketing|
|  4|  Diana|Engineering|
+---+-------+-----------+



In [6]:
spark.sql("""
    SELECT department, COUNT(*) as employee_count
    FROM polaris.demo.employees
    GROUP BY department
    ORDER BY employee_count DESC
""").show()

+-----------+--------------+
| department|employee_count|
+-----------+--------------+
|Engineering|             2|
|      Sales|             1|
|  Marketing|             1|
+-----------+--------------+



### Explore the Spark UI

**URL:** http://localhost:4040

Useful tabs: **Jobs** (overview), **Stages** (task breakdown), **SQL** (query plans), **Environment** (config). Good for performance tuning and debugging.

## Query with Trino

Iceberg's big win is **engine interoperability**: the table format is open, so any engine can read the same files. Let's query the same table from Trino (a separate SQL engine) over JDBC.

In [7]:
from trino.dbapi import connect
import pandas as pd
import time

# Connect to Trino (with retry for container startup timing)
max_retries = 3
for attempt in range(max_retries):
    try:
        conn = connect(
            host='trino',
            port=8080,
            user='trino',
            catalog='polaris',
            schema='demo'
        )

        cursor = conn.cursor()

        # Query the same table we created with Spark!
        cursor.execute("SELECT * FROM employees")
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]

        # Display results
        df = pd.DataFrame(rows, columns=columns)
        print(" Querying via Trino:\n")
        print(df.to_string(index=False))

        cursor.close()
        conn.close()
        break
    except Exception as e:
        if attempt < max_retries - 1:
            print(f"Trino connection attempt {attempt + 1} failed, retrying in 5s...")
            time.sleep(5)
        else:
            raise

 Querying via Trino:

 id    name  department
  3 Charlie   Marketing
  4   Diana Engineering
  1   Alice Engineering
  2     Bob       Sales


### Explore the Trino UI

**URL:** http://localhost:8080 · **Credentials:** `admin`

Click any query (filter to *finished*) to see its execution plan, data scanned, and metrics. Also shows cluster overview and live queries.

## Explore MinIO

**URL:** http://localhost:9001/browser/warehouse/demo/employees · **Credentials:** `admin / password`

MinIO holds the actual files behind your Iceberg tables:

- `data/` — **Parquet** files (columnar binary; Iceberg's default storage format)
- `metadata/` — Iceberg's metadata tree

The metadata is layered for query skipping:

- **Snapshot** → points to a manifest list (full table state at a point in time)
- **Manifest list** → groups manifests with partition summaries (skip whole groups of files)
- **Manifest** → lists data files with partition values + column min/max stats (skip individual files)
- **Data files** → the Parquet files themselves

→ Deep dive in [`docs/02-metadata-tree.md`](../docs/02-metadata-tree.md). The next exercise (E1.2) shows this in action.

## Congrats!

You've set up an open lakehouse: catalog (Polaris) + storage (MinIO) + two engines (Spark, Trino) reading the same Iceberg tables.